In [2]:
import chess, chess.engine, os, stat
from policy import *
import random

2026-03-21 10:54:10.201404: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-21 10:54:10.203012: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-21 10:54:10.238377: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-21 10:54:10.238890: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-21 10:54:11.488592: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Co

In [3]:
from stockfish import Stockfish
engine_path = r"./stockfish/src/stockfish"
sf = Stockfish(engine_path, parameters={"Threads": 1, "Hash": 256})
sf.set_depth(2)           
sf.set_skill_level(2)
sf.get_engine_parameters()

{'Debug Log File': '',
 'Contempt': 0,
 'Min Split Depth': 0,
 'Ponder': False,
 'MultiPV': 1,
 'Skill Level': 2,
 'Move Overhead': 10,
 'Minimum Thinking Time': 20,
 'Slow Mover': 100,
 'UCI_Chess960': False,
 'UCI_LimitStrength': False,
 'UCI_Elo': 1350,
 'Threads': 1,
 'Hash': 256}

In [4]:
games= load_json("1400_1500_games.json")

In [5]:
agent = Agent("")
agent.train(games)

332480/332480 [==============================] - 28390s 85ms/step - loss: 3.1421 - accuracy: 0.2370


In [6]:
board = chess.Board()
def stockfish_move():
    sf.set_fen_position(board.fen())
    move = sf.get_best_move()
    board.push(chess.Move.from_uci(move))
def agent_move():
    move = agent.act(board)
    board.push(move)

In [7]:
NUM_GAMES = 1000
import json

games_data = []

for i in range(NUM_GAMES):

    board = chess.Board()
    moves = []

    while not board.is_game_over():

        if board.turn == chess.WHITE:
            move = agent.act(board)

        else:
            sf.set_fen_position(board.fen())
            best = sf.get_best_move()

            if best is None:
                break

            move = chess.Move.from_uci(best)

        board.push(move)
        moves.append(move.uci()) 

    game_data = {
        "event": "Agent vs Stockfish",
        "round": i + 1,
        "white": f"Mimic Agent of {agent.id}",
        "black": "Stockfish",
        "result": board.result(),
        "moves": moves
    }

    games_data.append(game_data)


safe_id = str(agent.id).replace(" ", "_").lower()
with open(f"{safe_id}_agent_vs_stockfish.json", "w") as f:
    json.dump(games_data, f, indent=4)